# bid WTP data extraction for figure 2

This script takes loops though all raw e-prime files and creates a database of WTP and onset data and is for use on Alliance Canada. 

## Install required packages

In [1]:
!pip install --no-index numpy pandas

In [2]:
import numpy as np
import pandas as pd
import os

In [3]:
bdd_bid = pd.DataFrame()

In [4]:


def bid_extract(directory_file, filename):
    colnames = ['Picture', 'Stim_Start_Time', 'Bid_Start_Time', 'Bid_Duration', 'Price']
    bid = pd.read_csv(os.path.join(directory_file, filename), sep='\t', header=None, names=colnames, skiprows=2, index_col=False)
    id_participant = filename[1:4]
    id_participant = int(id_participant)
 
    if id_participant <= 9:
        id_participant = 'RND00' + str(id_participant)
    elif 10 <= id_participant <= 95:
        id_participant = 'RND0' + str(id_participant)
    elif 96 <= id_participant <= 99:
        id_participant = 'RCX0' + str(id_participant)
    elif 100 <= id_participant <= 899:
        id_participant = 'RCX' + str(id_participant)
    elif id_participant >= 900:
        id_participant = 'RGC' + str(id_participant)
  
    session = filename[0:1]
    if session == '1':
        session = 1
    elif session == '2' and id_participant.startswith('RGC'):
        session = 'x'
    elif session == '2':
        session = 2
    elif session == '3' and id_participant.startswith('RGC'):
        session = 2
    elif session == '3':
        session = 3
    elif session == '4':
        session = 4
    else:
        session = 'erreur'
    
    run = int(filename[5:6])
      
    bid.insert(0, 'id_participant', id_participant)
    bid.insert(1, 'session', session)
    bid.insert(2, 'run', run)
    
    # Convert columns to numeric, coercing errors
    bid['Bid_Start_Time'] = pd.to_numeric(bid['Bid_Start_Time'], errors='coerce')
    bid['Stim_Start_Time'] = pd.to_numeric(bid['Stim_Start_Time'], errors='coerce')
    bid['Bid_Duration'] = pd.to_numeric(bid['Bid_Duration'], errors='coerce')
    bid['Price'] = pd.to_numeric(bid['Price'], errors='coerce')

    # Handle NaN values if necessary (e.g., fill with 0 or drop rows)
    bid = bid.dropna(subset=['Bid_Start_Time', 'Stim_Start_Time'])

    # Perform the operations
    bid['duration'] = bid['Bid_Start_Time'] - bid['Stim_Start_Time']
    bid['onset_bid_end'] = bid['Bid_Start_Time'] + bid['Bid_Duration']

    # Ensure the split operation produces exactly two columns
    split_columns = bid['Picture'].str.split('/', expand=True, n=1)
    if split_columns.shape[1] == 2:
        bid[['stimuli', 'stim_file']] = split_columns
    else:
        bid['stimuli'] = split_columns[0] if not split_columns.empty else ''
        bid['stim_file'] = ''

    bid['trial_type'] = bid['stim_file'].str[:2]
    bid['stim_file'] = bid['stim_file'].str[2:]
    bid['trial_type2'] = bid["stim_file"].str.extract("(Sweet|Salt)")[0]
    bid['trial_type2'] = bid['trial_type2'].fillna('Lo')
    bid['stim_file'] = bid['stim_file'].str.replace('Sweet', '')
    bid['stim_file'] = bid['stim_file'].str.replace('Salt', '')
    bid.rename({'Stim_Start_Time': 'onset', 'Bid_Start_Time': 'onset_bid', 'Bid_Duration': 'response_time', 'Price': 'value'}, axis=1, inplace=True)

    # discontinued event logic
    # event = bid[['onset', 'duration', 'trial_type', 'trial_type2', 'onset_bid', 'onset_bid_end', 'response_time', 'bid', 'stim_file']]
  
   
    # directory = os.path.join('/lustre03/project/6051963/share/GutBrain/data/', f'sub-{id_participant}_ses-{session}_task-BDM_run-{run}_events.tsv')
    # event.to_csv(directory, sep='\t', index=False)
    
    return bid

In [5]:


root_dir = "/lustre09/project/6051963/share/GutBrain/data/sourcedata"

bdd_bid = pd.DataFrame()

for dirpath, _, filenames in os.walk(root_dir):
    for filename in sorted(filenames):
        # Keep actual run files; allow .txt or extensionless names if present
        is_txt = filename.lower().endswith(".txt")
        is_extless = "." not in filename

        if not (is_txt or is_extless):
            continue

        # Skip obvious non-run stimulus labels if they are present as files
        if filename.lower().startswith(("im_", "ps_", "xx_")):
            continue

        bid = bid_extract(dirpath, filename)

        if isinstance(bid, pd.DataFrame) and not bid.empty:
            bdd_bid = pd.concat([bdd_bid, bid], ignore_index=True, sort=False)

In [6]:
bdd_bid[['stimuli', 'picture']] = bdd_bid['Picture'].str.split('/', expand=True)
bdd_bid['picture'] = bdd_bid['picture'].map(lambda x: str(x)[:-4])
bdd_bid['calories'] = bdd_bid['picture'].str[:2]
bdd_bid['picture'] = bdd_bid['picture'].str[2:]
bdd_bid["Sweet_Salt_Lo"] = bdd_bid["picture"].str.extract("(Sweet|Salt)")[0] 
bdd_bid['Sweet_Salt_Lo'] = bdd_bid['Sweet_Salt_Lo'].fillna('Lo')
bdd_bid['picture'] = bdd_bid['picture'].str.replace('Sweet', '')
bdd_bid['picture'] = bdd_bid['picture'].str.replace('Salt', '')

KeyError: 'Picture'

In [ ]:
bdd_bid.columns

Index(['id_participant', 'session', 'run', 'Picture', 'onset', 'onset_bid',
       'response_time', 'value', 'duration', 'onset_bid_end', 'stimuli',
       'stim_file', 'trial_type', 'trial_type2', 'picture', 'calories',
       'Sweet_Salt_Lo'],
      dtype='str')

In [ ]:
bdd_bid = bdd_bid[[
    'id_participant',
    'session',
    'run',
    'calories',
    'Sweet_Salt_Lo',
    'picture',
    'onset',  # Previously 'Stim_Start_Time'
    'onset_bid',  # Previously 'Bid_Start_Time'
    'response_time',  # Previously 'Bid_Duration'
    'value'  # Previously 'Price'
]]

In [ ]:
outdir = '/lustre09/project/6051963/share/GutBrain/results/BDD'
os.makedirs(outdir, exist_ok=True)
bdd_bid.to_csv(os.path.join(outdir, 'GB_bid_fmri.csv'), index=False)

## Script QC

manuel check for the combinations below

In [ ]:
sample = (
    bdd_bid[['id_participant', 'session', 'run', 'picture', 'value']]
    .drop_duplicates()
    .sample(n=20, random_state=42)
    .reset_index(drop=True)
)

print(sample)

   id_participant session  run          picture  value
0          RND032       3    1          Raisins    3.5
1          RCX203       1    1         Maynards    2.0
2          RND022       1    2          Activia    3.5
3          RCX247       4    1          Amandes    3.0
4          RND071       4    3        Goldfish2    4.0
5          RCX096       3    1            mandm    1.5
6          RND047       1    2           Orange    2.5
7          RND052       3    2         Tostitos    2.5
8          RND053       1    1        Redpepper    2.0
9          RND012       4    3          Amandes    4.0
10         RND027       2    2             Twix    1.5
11         RND003       1    3             Lays    1.5
12         RGC925       1    3    Hersheykisses    2.0
13         RGC905       1    2         Werthers    0.5
14         RND050       4    3          Activia    1.5
15         RGC920       x    2  Maynardspeaches    2.0
16         RND010       3    3          Raisins    3.0
17        

manuel check for the combinations below for Nans

In [ ]:
cols = ['picture', 'onset', 'onset_bid', 'response_time', 'value']
nan_mask = bdd_bid[cols].isna().any(axis=1)

sample_nan = (
    bdd_bid.loc[nan_mask, ['id_participant', 'session', 'run', *cols]]
    .sample(n=min(20, nan_mask.sum()), random_state=42)
    .reset_index(drop=True)
)

print(sample_nan)

   id_participant session  run            picture    onset  onset_bid  \
0          RGC906       2    2     Tomatescerises  291.126    295.138   
1          RCX148       2    1            Amandes  112.441    116.468   
2          RCX148       1    2           Snickers   44.615     48.637   
3          RND044       3    1           pringles  243.832    247.859   
4          RND060       3    1           Pretzels  495.742    499.759   
5          RND024       4    1          Dairymilk  332.994    337.021   
6          RND035       3    3              mandm  319.838    323.862   
7          RGC917       2    2     RufflesRegular  254.973    258.991   
8          RGC921       2    2               Oreo  279.404    283.413   
9          RGC907       x    1           Brownies  475.659    479.688   
10         RGC907       x    1             Fritos   93.756     97.785   
11         RCX247       2    1             banana   30.991     35.010   
12         RND005       1    3              Apple  